## Outlier detection
For that, the constraint that matters is that a single bad point contaminates its own neighborhood, and with ~20 observations you can't afford to fit a model that the outlier itself distorts. So: work on **year-over-year differences**, use a **robust scale**, and **classify the shape** of what you flag.

**Why differences.** Levels are trending, so an outlier in levels is confounded with the trend. Differencing removes level and (locally) slope, and it gives you a diagnostic signature for free: an isolated bad value produces a large jump followed immediately by a nearly equal jump in the *opposite* direction. A genuine level shift produces one large jump and then nothing. Same magnitude, completely different meaning — a spike is usually a data error, a level shift is usually a real event or a methodology change.

**Center before you standardize.** A country growing steadily at +2/yr should not have its normal years flagged. So z = (dₜ − median of that country's differences) / robust scale.

**The scale estimate is the fragile part.** With ~20 differences, MAD has poor efficiency and it hits exactly zero whenever the index is sticky or coarsely rounded — which then makes every nonzero change look infinitely extreme. Three fixes, all needed: use Qn instead of MAD (82% efficiency vs 37%, and far better behaved at n=20); floor the scale at the reporting granularity; and **partially pool toward the income-group median scale**, weight nc/(nc+m). That's the real use of your income buckets here — volatility and measurement error both differ systematically across them, so the group gives you a sensible prior for how noisy a country *should* be.

**Then recalibrate at the panel level.** Small-sample robust scales are biased low, so raw z's run hot and a 3.5 cutoff over-flags. Take the MAD of all z's across the whole panel (the vast majority are null), and divide through by it. Now 3.5 means what you think it means.

**Two things the per-country view can't see.** Gaps in the panel — a difference spanning three years isn't comparable to a one-year difference, so exclude those from scale estimation and mark them separately. And year-clustering: if 40 countries flag in 2011, that's the index revising its methodology, not 40 simultaneous national events. Always check the flag count by year before you believe any individual flag.

The endpoints are genuinely unidentifiable: a spike in the first year and a level shift in the second produce identical data. Flag them, mark them ambiguous, eyeball them.

Here's the whole thing implemented:Two fixes — a spike creates two large differences (the jump out and the jump back), and the return trip was being reported as a separate "level_shift". Let me collapse that:Verified on synthetic data: all 9 injected spikes and all 6 injected level shifts were recovered with the correct label, plus 3 false positives out of ~1,100 country-years.Run it as `python trajectory_breaks.py` to see the demo, or point it at your panel:

```python
res = detect_trajectory_breaks(df, "iso3", "year", "index_value", "income_group")
```

Three things to look at before you trust any individual flag:

`res.year_summary` first, always. If one year flags far more countries than the rest, that's an index revision and those flags belong to the index, not the countries. `res.data_quality` next — flat runs and interpolated stretches don't trigger the z-test at all (they're too *smooth*), but they're often the bigger problem in index panels, and a country that's 40% carry-forward shouldn't be in your analysis regardless of whether it has spikes. Then `res.country_scales`, to check whether `own_scale` varies systematically by income group. If it doesn't, drop the pooling (`pool_m=0`) and simplify.

`pip install statsmodels` if you can — it enables the Qn estimator instead of the MAD fallback, which is a real accuracy gain at these sample sizes.

Two things I'd tune to your data. `z_thresh=3.5` is a starting point, not a law: plot a histogram of `res.flags.z` and see where the bulk ends — if there's a visible gap, put the threshold in it. And `min_obs=8` excludes your shortest series from this test entirely, which is correct; those countries need the cross-sectional approach instead, since there's no trajectory to break.

In [ ]:
"""
Within-country outlier detection for short annual index panels.

Detects country-years that break that country's OWN trajectory, using
gap-aware first differences + robust scale with partial pooling across
income groups. Designed for series of ~10-25 observations, where
per-country time-series models (ARIMA, STL, changepoint) overfit.

Usage
-----
    import pandas as pd
    from trajectory_breaks import detect_trajectory_breaks, plot_country

    res = detect_trajectory_breaks(
        df, country_col="iso3", year_col="year",
        value_col="index_value", group_col="income_group",
    )
    res.flags.query("flag")            # flagged country-years
    res.year_summary                   # check for methodology breaks
    res.data_quality                   # flat runs / interpolation
    plot_country(res, "BRA")

Requires numpy + pandas. statsmodels optional (better scale estimator).
matplotlib optional (only for plot_country).
"""

from dataclasses import dataclass
from typing import Optional
import numpy as np
import pandas as pd

__all__ = ["detect_trajectory_breaks", "plot_country", "BreakResult"]


# --------------------------------------------------------------------------
# robust scale
# --------------------------------------------------------------------------

def robust_scale(x: np.ndarray) -> float:
    """Qn scale estimator, falling back to MAD then normalized IQR.

    Qn is preferred: ~82% Gaussian efficiency vs ~37% for MAD, and it does
    not assume symmetry about a centre. It matters a lot at n=20.
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size < 3:
        return np.nan

    s = np.nan
    try:
        from statsmodels.robust.scale import qn_scale
        s = float(qn_scale(x))
    except Exception:
        pass

    if not np.isfinite(s) or s <= 0:
        s = 1.4826 * float(np.median(np.abs(x - np.median(x))))
    if not np.isfinite(s) or s <= 0:
        q1, q3 = np.percentile(x, [25, 75])
        s = (q3 - q1) / 1.349
    return float(s) if np.isfinite(s) and s > 0 else np.nan


def _infer_granularity(values: np.ndarray) -> float:
    """Guess the reporting granularity (rounding step) of the index.

    Used to floor the scale estimate so that coarsely-rounded or sticky
    series don't produce astronomically large z-scores.
    """
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return 1e-9
    # decimal places actually used, capped at 6
    decs = 0
    for d in range(7):
        if np.allclose(v, np.round(v, d), atol=0, rtol=0):
            decs = d
            break
    else:
        decs = 6
    return float(10.0 ** (-decs))


# --------------------------------------------------------------------------
# result container
# --------------------------------------------------------------------------

@dataclass
class BreakResult:
    flags: pd.DataFrame        # one row per country-year transition
    country_scales: pd.DataFrame
    year_summary: pd.DataFrame
    data_quality: pd.DataFrame
    series: pd.DataFrame       # cleaned input, for plotting
    calibration: float
    params: dict


# --------------------------------------------------------------------------
# main
# --------------------------------------------------------------------------

def detect_trajectory_breaks(
    df: pd.DataFrame,
    country_col: str = "country",
    year_col: str = "year",
    value_col: str = "value",
    group_col: Optional[str] = None,
    min_obs: int = 8,
    pool_m: float = 10.0,
    z_thresh: float = 3.5,
    granularity: Optional[float] = None,
    calibrate: bool = True,
) -> BreakResult:
    """Flag country-years that break the country's own trajectory.

    Parameters
    ----------
    group_col : income group (or any peer grouping). Used to pool the
        volatility estimate for short series. Pass None to pool globally.
    min_obs : countries with fewer usable observations are not evaluated
        here -- use cross-sectional detection for those instead.
    pool_m : pseudo-count for shrinking a country's own scale toward its
        group's. Higher = more pooling. 10 is a reasonable default for
        15-25 year series.
    z_thresh : cutoff on the calibrated robust z. 3.5 is conventional.
    granularity : reporting step of the index (e.g. 0.1). Inferred if None.
    calibrate : rescale z panel-wide to correct small-sample bias in the
        per-country scale estimates. Recommended.
    """
    d = df[[c for c in {country_col, year_col, value_col, group_col} if c]].copy()
    d = d.dropna(subset=[country_col, year_col, value_col])
    d[year_col] = d[year_col].astype(int)
    d[value_col] = d[value_col].astype(float)
    d = d.sort_values([country_col, year_col]).reset_index(drop=True)
    d = d.drop_duplicates(subset=[country_col, year_col], keep="last")

    if granularity is None:
        granularity = _infer_granularity(d[value_col].values)
    scale_floor = 0.75 * granularity

    # ---- build the difference table -------------------------------------
    rows = []
    quality = []
    n_obs = d.groupby(country_col)[value_col].size()

    for country, g in d.groupby(country_col, sort=False):
        g = g.sort_values(year_col)
        yrs = g[year_col].to_numpy()
        vals = g[value_col].to_numpy()
        grp = g[group_col].iloc[-1] if group_col else "_all_"

        gaps = np.diff(yrs)
        rates = np.diff(vals) / gaps          # per-year change

        for i in range(len(rates)):
            rows.append({
                country_col: country,
                year_col: int(yrs[i + 1]),     # year the change lands on
                "prev_year": int(yrs[i]),
                "group": grp,
                "value": vals[i + 1],
                "prev_value": vals[i],
                "gap": int(gaps[i]),
                "d": rates[i],
                "n_obs": len(yrs),
            })

        # ---- data-quality screens (independent of the outlier test) ------
        # flat runs: 3+ identical consecutive values => likely carry-forward
        run_start, run_len = 0, 1
        for i in range(1, len(vals)):
            if vals[i] == vals[i - 1] and gaps[i - 1] == 1:
                run_len += 1
            else:
                if run_len >= 3:
                    quality.append({country_col: country, "issue": "flat_run",
                                    "start_year": int(yrs[run_start]),
                                    "end_year": int(yrs[run_start + run_len - 1]),
                                    "length": run_len, "detail": f"value={vals[run_start]}"})
                run_start, run_len = i, 1
        if run_len >= 3:
            quality.append({country_col: country, "issue": "flat_run",
                            "start_year": int(yrs[run_start]),
                            "end_year": int(yrs[run_start + run_len - 1]),
                            "length": run_len, "detail": f"value={vals[run_start]}"})

        # linear interpolation: 3+ consecutive identical nonzero yearly steps
        if len(rates) >= 3:
            j = 0
            while j < len(rates) - 2:
                k = j
                while (k + 1 < len(rates)
                       and gaps[k] == 1 and gaps[k + 1] == 1
                       and rates[k + 1] != 0
                       and np.isclose(rates[k + 1], rates[j], rtol=1e-12, atol=1e-12)):
                    k += 1
                if k - j + 1 >= 3:
                    quality.append({country_col: country, "issue": "linear_interpolation",
                                    "start_year": int(yrs[j]), "end_year": int(yrs[k + 1]),
                                    "length": k - j + 2, "detail": f"step={rates[j]:.6g}"})
                    j = k + 1
                else:
                    j += 1

    diffs = pd.DataFrame(rows)
    if diffs.empty:
        raise ValueError("No usable consecutive observations found.")

    # ---- per-country centre and scale (unit-gap differences only) --------
    unit = diffs[diffs["gap"] == 1]

    centres, scales = {}, {}
    for country, g in unit.groupby(country_col, sort=False):
        centres[country] = float(np.median(g["d"]))
        scales[country] = robust_scale(g["d"].to_numpy()) if len(g) >= 5 else np.nan

    cs = pd.DataFrame({
        country_col: list(centres.keys()),
        "centre": [centres[c] for c in centres],
        "own_scale": [scales[c] for c in centres],
    })
    cs["n_obs"] = cs[country_col].map(n_obs)
    cs["n_diffs"] = cs[country_col].map(unit.groupby(country_col).size())
    cs["group"] = cs[country_col].map(
        diffs.groupby(country_col)["group"].last()
    )

    # ---- pool the scale toward the group (this is what income groups buy) -
    grp_scale = cs.groupby("group")["own_scale"].median()
    global_scale = float(np.nanmedian(cs["own_scale"]))
    cs["group_scale"] = cs["group"].map(grp_scale).fillna(global_scale)
    cs["group_scale"] = cs["group_scale"].fillna(global_scale)

    w = cs["n_diffs"].fillna(0) / (cs["n_diffs"].fillna(0) + pool_m)
    own = cs["own_scale"].fillna(cs["group_scale"])
    cs["scale"] = (w * own + (1 - w) * cs["group_scale"]).clip(lower=scale_floor)

    # ---- z-scores --------------------------------------------------------
    diffs = diffs.merge(cs[[country_col, "centre", "scale", "own_scale",
                            "group_scale", "n_diffs"]],
                        on=country_col, how="left")
    diffs["d_centred"] = diffs["d"] - diffs["centre"]
    diffs["z_raw"] = diffs["d_centred"] / diffs["scale"]

    evaluable = (diffs["gap"] == 1) & (diffs["n_obs"] >= min_obs)

    calib = 1.0
    if calibrate and evaluable.sum() >= 50:
        zz = diffs.loc[evaluable, "z_raw"].to_numpy()
        mad = 1.4826 * np.median(np.abs(zz - np.median(zz)))
        if np.isfinite(mad) and mad > 0:
            calib = float(mad)
    diffs["z"] = diffs["z_raw"] / calib

    diffs["evaluated"] = evaluable
    diffs["flag"] = evaluable & (diffs["z"].abs() > z_thresh)

    # ---- classify the shape of each flagged break ------------------------
    diffs = diffs.sort_values([country_col, year_col]).reset_index(drop=True)
    nxt_z = diffs.groupby(country_col)["z"].shift(-1)
    nxt_dc = diffs.groupby(country_col)["d_centred"].shift(-1)
    nxt_gap = diffs.groupby(country_col)["gap"].shift(-1)
    is_last = diffs.groupby(country_col).cumcount(ascending=False) == 0

    def classify(i):
        if not diffs.at[i, "flag"]:
            return ""
        if is_last[i]:
            return "ambiguous_endpoint"
        if nxt_gap[i] != 1 or not np.isfinite(nxt_dc[i]):
            return "ambiguous_gap"
        dc, nd = diffs.at[i, "d_centred"], nxt_dc[i]
        if dc == 0:
            return "complex"
        rev = -nd / dc                      # 1.0 = full reversal
        if rev >= 0.6:
            return "spike"                  # single bad value; suspect error
        if rev < 0.25 and abs(nxt_z[i]) < 2.5:
            return "level_shift"            # real event or methodology change
        return "complex"

    diffs["break_type"] = [classify(i) for i in range(len(diffs))]

    # A spike shows up twice: the jump out and the jump back. The second row
    # is the same event, so demote it rather than reporting two breaks.
    prev_type = diffs.groupby(country_col)["break_type"].shift(1)
    prev_dc = diffs.groupby(country_col)["d_centred"].shift(1)
    recovery = (diffs["flag"]
                & (prev_type == "spike")
                & (diffs["gap"] == 1)
                & (np.sign(diffs["d_centred"]) != np.sign(prev_dc)))
    diffs.loc[recovery, "break_type"] = "spike_recovery"
    diffs.loc[recovery, "flag"] = False

    # ---- year clustering: is it the country, or is it the index? ---------
    ys = (diffs[diffs["evaluated"]]
          .groupby(year_col)
          .agg(n_countries=("flag", "size"), n_flagged=("flag", "sum")))
    ys["share_flagged"] = ys["n_flagged"] / ys["n_countries"]
    base = ys["share_flagged"].median()
    ys["excess_vs_median"] = ys["share_flagged"] - base
    ys["suspect_index_break"] = ys["share_flagged"] > max(3 * base, 0.05)
    ys = ys.reset_index()

    quality_df = pd.DataFrame(quality) if quality else pd.DataFrame(
        columns=[country_col, "issue", "start_year", "end_year", "length", "detail"])

    out_cols = [country_col, year_col, "prev_year", "group", "gap",
                "prev_value", "value", "d", "d_centred", "z",
                "flag", "break_type", "evaluated", "n_obs", "scale"]

    return BreakResult(
        flags=diffs[out_cols],
        country_scales=cs,
        year_summary=ys,
        data_quality=quality_df,
        series=d.rename(columns={country_col: country_col, year_col: year_col,
                                 value_col: "value"}),
        calibration=calib,
        params=dict(country_col=country_col, year_col=year_col,
                    value_col=value_col, group_col=group_col,
                    min_obs=min_obs, pool_m=pool_m, z_thresh=z_thresh,
                    granularity=granularity, scale_floor=scale_floor),
    )


# --------------------------------------------------------------------------
# inspection
# --------------------------------------------------------------------------

def plot_country(res: BreakResult, country: str, ax=None):
    """Plot one country's series with flagged years marked. Triage tool --
    always eyeball a flag before acting on it."""
    import matplotlib.pyplot as plt

    cc, yc = res.params["country_col"], res.params["year_col"]
    s = res.series[res.series[cc] == country].sort_values(yc)
    f = res.flags[(res.flags[cc] == country) & res.flags["flag"]]

    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4))
    ax.plot(s[yc], s["value"], "o-", lw=1.4, ms=4, color="#444")

    colors = {"spike": "#d62728", "level_shift": "#ff7f0e",
              "complex": "#9467bd", "ambiguous_endpoint": "#7f7f7f",
              "ambiguous_gap": "#7f7f7f"}
    seen = set()
    for _, r in f.iterrows():
        c = colors.get(r["break_type"], "#d62728")
        lab = r["break_type"] if r["break_type"] not in seen else None
        seen.add(r["break_type"])
        ax.axvline(r[yc], color=c, alpha=0.35, lw=6, label=lab)
        ax.annotate(f"z={r['z']:.1f}", (r[yc], r["value"]),
                    textcoords="offset points", xytext=(4, 8), fontsize=8, color=c)

    ax.set_title(f"{country}  (n={len(s)})")
    ax.set_xlabel(yc)
    ax.set_ylabel("index")
    if seen:
        ax.legend(fontsize=8, frameon=False)
    ax.grid(alpha=0.25)
    return ax


# --------------------------------------------------------------------------
if __name__ == "__main__":
    # synthetic demo: 60 countries, 20 years, with injected spikes/shifts
    rng = np.random.default_rng(0)
    recs = []
    for i in range(60):
        base = rng.uniform(30, 80)
        trend = rng.normal(0.4, 0.4)
        sd = rng.choice([0.4, 0.9, 1.8])
        v = base + np.cumsum(rng.normal(trend, sd, 20))
        if i % 7 == 0:
            v[rng.integers(3, 17)] += rng.choice([-1, 1]) * 9 * sd   # spike
        if i % 11 == 0:
            v[rng.integers(5, 15):] += rng.choice([-1, 1]) * 8 * sd  # level shift
        grp = ["low", "lower_mid", "upper_mid", "high"][i % 4]
        for t, val in zip(range(2000, 2020), v):
            recs.append(dict(country=f"C{i:02d}", year=t, value=round(val, 1),
                             income_group=grp))

    demo = pd.DataFrame(recs)
    res = detect_trajectory_breaks(demo, "country", "year", "value", "income_group")
    print(f"calibration factor: {res.calibration:.2f}")
    print(res.flags.query("flag")[["country", "year", "z", "break_type"]]
          .to_string(index=False))
    print("\nyears with clustered flags (suspect index revision):")
    print(res.year_summary.query("suspect_index_break").to_string(index=False))